# Geospatial Filtering and Indexing

Author: [Koussay Mehdouani](https://github.com/amkkoussay)

Questions about cutting a gridded dataset down to a place come up all the
time: *How do I extract one region? How do I keep only the grid cells near a
point? How do I read the values at my weather stations?* This example works
through those three questions with label-based indexing (``sel``), masking
(``where``), and vectorized selection, and finishes with a note on aligning
two grids in space.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

## The dataset

We use the ``air_temperature`` tutorial dataset: 4x-daily air temperature on a
regular latitude/longitude grid over North America. Two details of its
coordinates matter for everything below:

- ``lat`` runs **from 75 down to 15** — it is a *descending* coordinate.
- ``lon`` runs from 200 to 330, i.e. it uses the **0 to 360** convention
  rather than -180 to 180.

In [ ]:
ds = xr.tutorial.load_dataset("air_temperature")
ds

## 1. Cut out a region of interest with ``sel`` and ``slice``

Label-based selection with ``sel`` is the standard way to extract a
rectangular region. One rule to remember: **the bounds of a slice must
follow the order of the coordinate itself**. Because ``lat`` descends here,
the slice for 40–50°N is ``slice(50, 40)`` — writing ``slice(40, 50)``
would return an empty selection.

Our region of interest is the Pacific Northwest, roughly 40–50°N and
235–250°E (i.e. 110–125°W in the -180 to 180 convention):

In [ ]:
region = ds.sel(lat=slice(50, 40), lon=slice(235, 250))
print(region.sizes)

region["air"].mean("time").plot()
plt.title("Mean air temperature, 40-50N, 235-250E");

## 2. Keep a non-rectangular area with ``where(..., drop=True)``

``sel`` can only cut rectangles. When the area of interest is defined by a
*condition* instead — say, every grid cell within 7.5 degrees of a center
point — use ``where``. The condition below is a 2D boolean array built by
broadcasting the 1D coordinates against each other.

Without ``drop=True``, ``where`` keeps the full grid and fills everything
outside the condition with ``NaN``. With ``drop=True``, the dimensions are
also trimmed to the smallest box that contains the selected cells (cells
inside that box but outside the circle are still ``NaN``):

In [ ]:
center_lat, center_lon = 47.5, 237.5  # near Seattle, in dataset coords
radius = 7.5

distance = np.sqrt((ds["lat"] - center_lat) ** 2 + (ds["lon"] - center_lon) ** 2)
near = ds.where(distance <= radius, drop=True)
print("full grid:", dict(ds.sizes))
print("after where + drop:", dict(near.sizes))

near["air"].mean("time").plot()
plt.title("Mean air temperature within 7.5 degrees of the center point");

## 3. Sample the grid at station locations

To read a gridded dataset at a list of points — weather stations, cities,
field sites — pass one ``DataArray`` per coordinate to ``sel``, all sharing
a new dimension (here ``station``). Xarray then performs a *vectorized*
lookup and returns one value per point. Our station longitudes are given in
the familiar -180 to 180 convention, so we convert them to the dataset's
0 to 360 convention first (``+ 360``). ``method="nearest"`` snaps each
station to the closest grid cell, which is 2.5 degrees away at most here:

In [ ]:
stations = pd.DataFrame(
    {
        "station": ["Seattle", "Denver", "Chicago"],
        "lat": [47.61, 39.74, 41.88],
        "lon": [-122.33, -104.99, -87.63],  # -180 to 180 convention
    }
)
stations["lon"] = stations["lon"] % 360  # convert to the dataset convention

sampled = ds.sel(
    lat=xr.DataArray(stations["lat"], dims="station"),
    lon=xr.DataArray(stations["lon"], dims="station"),
    method="nearest",
)
sampled = sampled.assign_coords(station=("station", stations["station"]))
sampled["air"]

The result is a time series per station, ready to plot or to export
(e.g. with ``sampled.to_dataframe()``) for comparison against station
observations:

In [ ]:
sampled["air"].isel(time=slice(0, 120)).plot.line(x="time", hue="station")
plt.ylabel("Air temperature (K)")
plt.title("First 30 days of 2013 at the three stations");

## 4. A note on aligning two grids in space

When two products live on *different* grids, values can be moved onto a
common grid with ``interp_like``, which interpolates the source onto the
target's coordinates. Here we pretend a second product only exists on every
other grid point, and interpolate it back onto the full grid:

In [ ]:
coarse = ds.isel(lat=slice(None, None, 2), lon=slice(None, None, 2))
aligned = coarse.interp_like(ds)
print("coarse:", dict(coarse.sizes))
print("aligned:", dict(aligned.sizes))

## Going further

- **Resampling to a new resolution / regridding** between arbitrary grids is
  a bigger topic; the dedicated tool in this ecosystem is
  [xESMF](https://xesmf.readthedocs.io), which builds regridding weights
  from the grid geometry.
- For raster data with coordinate reference systems (CRS) and reprojection,
  see [rioxarray](https://corteva.github.io/rioxarray/stable/).